# LayoutLM Document QA (impira) — DIMER E2E document question-answering fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/layoutlm-document-qa-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/layoutlm-document-qa-pipeline/blob/main/tutorials/layoutlm_document_qa_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-impira%2Flayoutlm--document--qa-ffcc4d?style=flat)](https://huggingface.co/impira/layoutlm-document-qa) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2Funilm%20(layoutlm)-181717?style=flat&logo=github&logoColor=white)](https://github.com/microsoft/unilm/tree/master/layoutlm) [![arXiv](https://img.shields.io/badge/arXiv-1912.13318-b31b1b.svg)](https://arxiv.org/abs/1912.13318)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** extractive document question answering over OCR words and boxes and bounded supervised fine-tuning of the last encoder blocks and the span head on a gold-span receipt dataset, using the pinned `impira/layoutlm-document-qa` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/layoutlm_document_qa_pipeline/`, at revision `7a9a15075dd1`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `beed3c4d02d86017ebca5bd0fdf210046b907aa6` (~514 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned `impira/layoutlm-document-qa` snapshot (safetensors, 511 MB), reads the `ground_truth` column of two digest-pinned CORD-v2 receipt shards from the Hugging Face Hub (about 0.4 MB over HTTP range requests, no credential), turns the 199 unique receipts into templated question/answer records over their real OCR words and boxes and cuts them by receipt into 119 / 30 / 50 training, validation and test pages, drops any record that would not fit one 512-token window, answers five authored questions over a rendered invoice through the inference contract with an input manifest and a rejection probe, scores the frozen model on the test receipts with ANLS and exact match beside the last-number and keyword-lookup baselines, runs a bounded fine-tuning of the last four encoder blocks and the span head on the training receipts with validation-ANLS epoch selection, scores the held-out receipts again, re-reads the invoice with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify answer parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about twenty minutes of model time after the downloads; a CUDA runtime is used automatically when present.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own labelled pages as a JSON array or a JSONL file of `{{id, page_id, question, words, boxes, image_size, answer_start, answer_end}}` records — the page's OCR words with one pixel `[x0, y0, x1, y1]` box each, the page size, and the inclusive word indices of the gold span. They pass through the same validation, seeded page-disjoint split, fit check, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the CORD-v2 sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

`impira/layoutlm-document-qa` is Impira's LayoutLM (v1) — a 12-layer BERT-style encoder with 2-D position embeddings, 128 M parameters, RoBERTa vocabulary — fine-tuned on SQuAD 2.0 and DocVQA for extractive document question answering and published under the **MIT** licence. At inference the encoder reads the question and the page's **words with their boxes** (normalised to a 0–1000 grid) as one token sequence and emits a start and an end logit per token; the carried module scores every span of at most 15 tokens by the product of the start and end softmax probabilities and returns the best one as a word span with its indices. **The model never sees pixels:** OCR is an input provider outside the model, so words and boxes come from the caller (bring-your-own OCR), and the model has no abstention — it returns its best span for every question.

What this notebook adds to inference is **adaptation with gold spans**. The dataset is real: CORD-v2 (Park et al., 2019; NAVER Clova; **CC BY 4.0**) — photographed Indonesian receipts whose every printed word carries a pixel box and a field category. Because the model needs words and boxes only, the notebook reads just the `ground_truth` column of two pinned Hub parquet shards (about 0.4 MB of the 476 MB the shards hold with their images) and turns each receipt into templated questions — *What is the total amount?*, *How much change was given?*, *What is the name of the first item?* — whose gold answer is the value words of the matching field, a contiguous span of the page's OCR words by construction. The frozen model already reads most receipt totals (the build record measured ANLS 0.84 on the test receipts), so the fine-tuning question is narrower and more honest than a rescue: does a bounded adaptation of the last four encoder blocks and the span head on 119 receipts lift the fields it gets wrong — item names, quantities, cash and change lines — on held-out receipts? Two metrics are implemented in the carried modules (mean **ANLS**, DocVQA's official normalised-Levenshtein score, and the **exact-match** rate after the same normalisation), and two **non-neural baselines** — last number on the page and keyword lookup — show where a reader with no model sits. Nothing here is a quality claim about your documents: it is one seeded split of one corpus with one OCR convention.

**Snapshot note:** the pinned revision ships a fast `tokenizer.json` (an 8-file manifest) and a float32 `model.safetensors` — no pickle is opened anywhere in this notebook; the upstream `pytorch_model.bin` and `tf_model.h5` are neither listed nor fetched. Section 3 stages and digest-verifies those eight files before the tokenizer or the model is constructed.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, metrics and dataset modules guarantee; stage and digest-verify the immutable upstream snapshot; fetch one column of a digest-pinned receipt corpus without downloading its images, turn it into gold-span records and validate and split it by page without leakage; drop the records the window ceiling would refuse rather than truncating them; answer through the public API over a rendered page and read `answer`, `start`, `end` and `score` correctly (a product of softmax masses, not a calibrated probability); score the frozen model against gold spans beside two non-neural baselines; run a bounded fine-tuning with explicit hyperparameters and validation-based epoch selection; evaluate on a page-disjoint test split; re-read a page from a different document family with the adapted model; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** OCR itself (the notebook installs no Tesseract and ships no OCR model; the default path uses the corpus's annotated words and boxes and the invoice renderer's own boxes), PDF or multi-page documents (one page per call), abstention (the model always returns its best span, even for an unanswerable question), answers that are not a contiguous span of the OCR words, arithmetic or reasoning, evaluation on the DocVQA benchmark (registration-gated and not bundled), full-model or embedding fine-tuning, training on pages that need more than one window, languages and scripts other than the Latin-script receipts and English questions used here, and any claim that a CORD-v2 split stands in for your documents. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available. CPU is adequate but not fast: the build record measured about 7 s to load and digest-verify the 514 MB snapshot, about 41 s to score the 229 test questions, and about two and a half minutes per epoch of fine-tuning the last four encoder blocks and the span head on 595 training questions (validation scoring included; 934 s for six epochs in the build record). The pinned `torch==2.14.0` install and the 511 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what extractive (span) question answering over OCR tokens is; why a start/end softmax product is not a calibrated confidence; what normalised Levenshtein similarity (ANLS) measures and why neither it nor exact match is a human judgement.
- **Data contract:** records are `{{id, page_id, question, words, boxes, image_size, answer_start, answer_end}}` — the page's OCR words (1..`MAX_WORDS` = 2,000, non-empty) with one pixel `[x0, y0, x1, y1]` box per word inside the page, the page size in pixels (`MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` = 1..10,000), a question of at most `MAX_QUESTION_CHARS` (256) characters, and the inclusive word indices of the gold span; an optional `answers` list must start with the span text. Ids match `[A-Za-z0-9_.:-]{{1,64}}` and are unique; a dataset needs 8..20,000 records; every question on the same page lands in the same split so a test page is never trained on; a question + page that needs more than one 512-token window is dropped from training by the fit check, never truncated. BYOD accepts a JSON array or JSONL in that shape.
- **Validation is structural, not semantic:** nothing checks that a question is answerable from its page beyond the gold span lying inside the words, or that a gold span is the *best* answer — a mislabelled corpus is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — an internal document set with its field annotations is exactly that. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path reads the `ground_truth` column of two objects in the Hub dataset repository `naver-clova-ix/cord-v2` at the immutable revision `7f0115a4…` (`data/test-…parquet`, 234,202,795 bytes, SHA-256 `51c65f17…`, and `data/validation-…parquet`, 242,080,800 bytes, SHA-256 `0d0f6dac…`): the declared size and SHA-256 of each file are checked against the pins before any byte is read, only the parquet footer and that one column are fetched over HTTPS range requests, and the decoded column is refused unless its own SHA-256 matches; the corpus is CC BY 4.0 (Park et al., 2019). No image is downloaded.
- **External access:** the Hugging Face Hub only, to fetch the pinned `impira/layoutlm-document-qa` snapshot (~514 MB in total) at revision `beed3c4d02d8…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
    'pyarrow==25.0.1',
]
NOTEBOOK_SOURCE = {
    'repository': 'layoutlm-document-qa-pipeline',
    'repository_revision': '7a9a15075dd16300ba8245cc6d53d1d0d59e5372',
    'embedded_module': 'src/layoutlm_document_qa_pipeline/pipeline.py',
    'embedded_modules': ['src/layoutlm_document_qa_pipeline/pipeline.py', 'src/layoutlm_document_qa_pipeline/metrics.py', 'src/layoutlm_document_qa_pipeline/samples.py'],
    'module_sha256': 'fbcc47580420d3f80ada7ab373bcf8e5c0abe204ffdc6448e75ff860d2e9fddb',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/layoutlm_document_qa_pipeline/` @ `7a9a15075dd1`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/layoutlm_document_qa_pipeline/pipeline.py`

In [ ]:
"""Extractive document question answering with the pinned ``impira/layoutlm-document-qa`` checkpoint.

The class loads the tokenizer and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the LayoutLM architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed.

LayoutLM (v1) reads **words and their boxes**, not pixels: OCR is an input provider outside the model.
The pipeline therefore takes ``words``/``boxes`` from the caller (bring-your-own OCR) and offers
``ocr_words_with_tesseract`` as an optional adapter that is imported only when called.

The adaptation contract (``check_fit``, ``evaluate``, ``adapt``, ``save_artifact``, ``from_artifact``)
fine-tunes the last encoder blocks plus the span head on a validated ``{id, question, words, boxes,
image_size, answer_start, answer_end}`` dataset with validation-ANLS epoch selection and exports the trained
tensors as a safetensors adapter bound to the pinned base weights. The inference contract above is unchanged.
"""

from __future__ import annotations

import hashlib
import json
import math
import re
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "impira/layoutlm-document-qa"
MODEL_REVISION = "beed3c4d02d86017ebca5bd0fdf210046b907aa6"
MODEL_LICENSE = "mit"
MODEL_KEY = "layoutlm-document-qa"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHT_FILE = "model.safetensors"
WEIGHT_SHA256 = (
    "e4bbad3e4a1b5ae50c787b7afd6049a0bfa99fd823b50436e444e092ae2347b9"  # manifest digest of WEIGHT_FILE
)
PARAMETER_COUNT = 127_792_898
ENCODER_LAYERS = 12  # config.json num_hidden_layers
DEFAULT_TRAINABLE_ENCODER_LAYERS = 4  # the last four encoder blocks plus the span head (28,353,026 params)
MAX_EVAL_RECORDS = 2_000
MAX_RECORDS_FIT = 20_000  # check_fit accepts a whole dataset before it is split
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.layoutlm-document-qa.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"

# Encoding ceilings. The checkpoint's max_position_embeddings is 514 (RoBERTa layout: 512 usable
# tokens); longer documents are split into overlapping windows of MAX_SEQ_LEN with DOC_STRIDE overlap
# and the best-scoring span across windows is returned (the transformers document-question-answering
# pipeline's convention, as is MAX_ANSWER_TOKENS).
MAX_SEQ_LEN = 512
DOC_STRIDE = 128
MAX_ANSWER_TOKENS = 15
MAX_WORDS = 2000
MAX_QUESTION_CHARS = 256
# Box ceilings. Boxes are pixel xyxy in the page's coordinate frame and are normalised to the
# 0..1000 grid LayoutLM expects (max_2d_position_embeddings 1024).
MAX_IMAGE_SIDE = 10000
MIN_IMAGE_SIDE = 1
BOX_GRID = 1000
# ANLS (DocVQA's official metric): a normalised Levenshtein similarity below this threshold scores 0.
ANLS_THRESHOLD = 0.5
_PUNCT_RE = re.compile(r"[^\w\s]")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def normalize_answer(text: str) -> str:
    """DocVQA-style normalisation: lower-case, punctuation removed, whitespace collapsed."""
    return " ".join(_PUNCT_RE.sub(" ", text.lower()).split())


def _levenshtein(a: str, b: str) -> int:
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    return previous[-1]


def anls(prediction: str, golds: Sequence[str], *, threshold: float = ANLS_THRESHOLD) -> float:
    """Average Normalised Levenshtein Similarity for one question (Biten et al., ICDAR 2019).

    ``1 - lev(pred, gold) / max(len(pred), len(gold))`` over normalised strings, maximised over the
    accepted ``golds``; a similarity below ``threshold`` scores 0 so a near-miss is not rewarded.
    """
    if not golds:
        raise ValueError("golds must contain at least one accepted answer")
    pred = normalize_answer(prediction)
    best = 0.0
    for gold in golds:
        ref = normalize_answer(gold)
        longest = max(len(pred), len(ref))
        similarity = 1.0 if longest == 0 else 1.0 - _levenshtein(pred, ref) / longest
        best = max(best, similarity)
    return best if best >= threshold else 0.0


def exact_match(prediction: str, golds: Sequence[str]) -> bool:
    """Whether the normalised prediction equals any normalised accepted answer."""
    pred = normalize_answer(prediction)
    return any(pred == normalize_answer(gold) for gold in golds)


def normalize_box(box: Sequence[float], width: int, height: int) -> list[int]:
    """Pixel xyxy -> LayoutLM's 0..1000 grid (the transformers pipeline's normalize_bbox)."""
    x0, y0, x1, y1 = (float(v) for v in box)
    return [
        int(BOX_GRID * (x0 / width)),
        int(BOX_GRID * (y0 / height)),
        int(BOX_GRID * (x1 / width)),
        int(BOX_GRID * (y1 / height)),
    ]


def ocr_words_with_tesseract(image: Image.Image, *, lang: str = "eng") -> tuple[list[str], list[list[float]]]:
    """Optional OCR adapter: words and pixel xyxy boxes from Tesseract via ``pytesseract``.

    Neither ``pytesseract`` nor the Tesseract binary is part of this package's pinned runtime; the
    import happens here so the rest of the pipeline stays usable with any OCR the caller prefers.
    """
    try:
        import pytesseract
    except ImportError as exc:  # pragma: no cover - depends on the host
        raise RuntimeError("pytesseract is not installed; supply words and boxes from your own OCR") from exc
    data = pytesseract.image_to_data(image.convert("RGB"), lang=lang, output_type=pytesseract.Output.DICT)
    words: list[str] = []
    boxes: list[list[float]] = []
    for text, left, top, w, h in zip(
        data["text"], data["left"], data["top"], data["width"], data["height"], strict=True
    ):
        token = str(text).strip()
        if not token:
            continue
        words.append(token)
        boxes.append([float(left), float(top), float(left + w), float(top + h)])
    return words, boxes


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one question string plus the page's OCR words (list of str) with one pixel xyxy box per word "
        "and the page size (width, height) the boxes are expressed in; pixels are not read by the model"
    ),
    "words": [1, MAX_WORDS],
    "question_chars": [1, MAX_QUESTION_CHARS],
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "encoding": (
        f"<s> question </s></s> words </s> with RoBERTa byte-level BPE; boxes normalised to 0..{BOX_GRID}; "
        f"windows of {MAX_SEQ_LEN} tokens with {DOC_STRIDE}-token overlap when the words do not fit"
    ),
    "output": (
        f"the best word span of at most {MAX_ANSWER_TOKENS} tokens with its start/end word indices and the "
        "span score (softmax start x softmax end within the window)"
    ),
}


def _check_page(image_size: Any) -> tuple[int, int]:
    if (
        not isinstance(image_size, Sequence)
        or isinstance(image_size, str)
        or len(image_size) != 2
        or any(isinstance(v, bool) or not isinstance(v, int) for v in image_size)
    ):
        raise TypeError("image_size must be a (width, height) pair of ints")
    width, height = int(image_size[0]), int(image_size[1])
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return width, height


def _check_document(
    words: Any, boxes: Any, image_size: Any
) -> tuple[list[str], list[list[int]], tuple[int, int]]:
    """Raise TypeError/ValueError naming the first violated ceiling; return words, grid boxes, size."""
    width, height = _check_page(image_size)
    if isinstance(words, str) or not isinstance(words, Sequence):
        raise TypeError("words must be a sequence of str")
    if not 1 <= len(words) <= MAX_WORDS:
        raise ValueError(f"words has {len(words)} entries; expected 1..MAX_WORDS={MAX_WORDS}")
    if not all(isinstance(word, str) and word.strip() for word in words):
        raise ValueError("every word must be a non-empty str")
    if isinstance(boxes, str) or not isinstance(boxes, Sequence) or len(boxes) != len(words):
        raise ValueError(f"boxes must have one pixel xyxy box per word ({len(words)})")
    grid: list[list[int]] = []
    for index, box in enumerate(boxes):
        if isinstance(box, str) or not isinstance(box, Sequence) or len(box) != 4:
            raise ValueError(f"boxes[{index}] must be [x0, y0, x1, y1]")
        try:
            x0, y0, x1, y1 = (float(v) for v in box)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"boxes[{index}] must hold numbers") from exc
        if not (0 <= x0 <= x1 <= width and 0 <= y0 <= y1 <= height):
            raise ValueError(f"boxes[{index}] {list(box)} is not inside the {width}x{height} page")
        grid.append(normalize_box((x0, y0, x1, y1), width, height))
    return [str(word) for word in words], grid, (width, height)


def _check_question(question: Any) -> str:
    if not isinstance(question, str):
        raise TypeError("question must be a str")
    checked = " ".join(question.split())
    if not checked:
        raise ValueError("question must contain at least one non-whitespace character")
    if len(checked) > MAX_QUESTION_CHARS:
        raise ValueError(f"question has {len(checked)} chars > MAX_QUESTION_CHARS {MAX_QUESTION_CHARS}")
    return checked


def validate_inputs(
    words: Sequence[str],
    boxes: Sequence[Sequence[float]],
    questions: Sequence[str],
    *,
    image_size: Sequence[int],
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Every question is checked exactly as ``answer`` would check it; rejection is reported by raising,
    and a caller that wants the finding recorded catches the exception and stores ``str(exc)`` under
    ``findings``.
    """
    checked_words, _grid, size = _check_document(words, boxes, image_size)
    if isinstance(questions, str) or not isinstance(questions, Sequence) or not questions:
        raise TypeError("questions must be a non-empty sequence of str")
    checked = [_check_question(question) for question in questions]
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (answer takes one page)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[0] if names else "page-0", "size": list(size), "n_words": len(checked_words)}
        ],
        "questions": checked,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    golds: Sequence[Sequence[str]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``golds`` (one sequence of accepted answers per result, in order) the report carries the
    mean ``anls`` and the ``exact_match`` rate over the questions plus one per-question entry, verdict
    ``sample-sanity``; without golds it is ``not-measurable`` and says what labelled data would make
    the task measurable.
    """
    if not results:
        raise ValueError("results must contain at least one answer result")
    base = {
        "task": "question + OCR words/boxes -> extractive answer span (LayoutLM v1)",
        "score_semantics": (
            "score is the product of the start and end softmax probabilities of the chosen span within "
            "its window under the model's own head — a ranking signal over spans of this page, not a "
            "calibrated probability that the answer is right, and never a signal that the question is "
            "answerable; the model always returns its best span"
        ),
        "sample_kind": sample_kind,
        "n_questions": len(results),
        "scores": [float(result.get("score", 0.0)) for result in results],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if golds is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no accepted answers were supplied for the evaluated questions",
            "needs": (
                "question/answer pairs with accepted answers on pages from the deployment domain "
                "(DocVQA-style annotations) scored with ANLS, plus the OCR the deployment will really use; "
                "no such labelled set ships with this repository"
            ),
        }
    if len(golds) != len(results):
        raise ValueError(f"golds has {len(golds)} entries for {len(results)} results")
    per_question = []
    for result, accepted in zip(results, golds, strict=True):
        if isinstance(accepted, str) or not accepted:
            raise ValueError("each golds entry must be a non-empty sequence of accepted answers")
        prediction = str(result["answer"])
        per_question.append(
            {
                "question": result.get("question"),
                "prediction": prediction,
                "score": float(result.get("score", 0.0)),
                "golds": list(accepted),
                "anls": anls(prediction, accepted),
                "exact_match": exact_match(prediction, accepted),
            }
        )
    metrics = [
        {
            "id": "anls",
            "value": sum(entry["anls"] for entry in per_question) / len(per_question),
            "threshold": ANLS_THRESHOLD,
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed; max over golds",
            "estimation": f"{len(per_question)} question(s) on one page, no dispersion estimate",
        },
        {
            "id": "exact_match",
            "value": sum(entry["exact_match"] for entry in per_question) / len(per_question),
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed",
            "estimation": f"{len(per_question)} question(s) on one page, no dispersion estimate",
        },
    ]
    return {
        **base,
        "metrics": metrics,
        "per_question": per_question,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(per_question)} authored question(s) on one tutorial page whose words and boxes you "
            "rendered yourself; plumbing evidence, not a DocVQA benchmark"
        ),
        "needs": (
            "a labelled question/answer set on pages from the deployment domain with the deployment's own "
            "OCR for any accuracy claim; the DocVQA benchmark itself is registration-gated and not bundled"
        ),
    }


def _window_bboxes(
    encoding: Any, window: int, grid_boxes: Sequence[Sequence[int]], sep_id: int
) -> list[list[int]]:
    """One 0..1000 box per token of a window: word tokens take their word's box, separators `[1000]*4`,
    everything else (the question and padding) `[0]*4` — the transformers document-question-answering
    pipeline's convention."""
    bbox = []
    for input_id, sequence_id, word_id in zip(
        encoding["input_ids"][window].tolist(),
        encoding.sequence_ids(window),
        encoding.word_ids(window),
        strict=True,
    ):
        if sequence_id == 1:
            bbox.append(list(grid_boxes[word_id]))
        elif input_id == sep_id:
            bbox.append([BOX_GRID] * 4)
        else:
            bbox.append([0] * 4)
    return bbox


@dataclass
class LayoutLMDocumentQAPipeline:
    """``_runner(question, words, grid_boxes)`` returns ``{"start": int, "end": int, "score": float}``
    (word indices, inclusive) or ``{"start": None, ...}`` when no span could be selected.
    ``_count_windows(question, words)`` returns how many 512-token windows the pair needs (1 = fits); both
    are injectable so the offline tests run without the model."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"
    _count_windows: Callable[[str, list[str]], int] | None = field(default=None, repr=False)
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _tokenizer: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> LayoutLMDocumentQAPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoTokenizer, LayoutLMForQuestionAnswering

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(location, **common)
        if not tokenizer.is_fast:
            raise RuntimeError("a fast tokenizer is required for word_ids/sequence_ids; snapshot has none")
        model = LayoutLMForQuestionAnswering.from_pretrained(location, dtype=torch.float32, **common)
        model = model.eval().to(resolved_device)
        for param in model.parameters():
            param.requires_grad_(False)
        sep_id = tokenizer.sep_token_id

        def encode(question: str, words: list[str]) -> Any:
            return tokenizer(
                text=question.split(),
                text_pair=words,
                is_split_into_words=True,
                max_length=MAX_SEQ_LEN,
                stride=DOC_STRIDE,
                truncation="only_second",
                return_overflowing_tokens=True,
                return_token_type_ids=True,
                padding="max_length",
                return_tensors="pt",
            )

        def count_windows(question: str, words: list[str]) -> int:
            return int(encode(question, words)["input_ids"].shape[0])

        def runner(question: str, words: list[str], grid_boxes: list[list[int]]) -> dict[str, Any]:
            encoding = encode(question, words)
            n_windows = int(encoding["input_ids"].shape[0])
            best: dict[str, Any] = {"start": None, "end": None, "score": 0.0, "n_windows": n_windows}
            model_device = next(model.parameters()).device
            for window in range(n_windows):
                sequence_ids = encoding.sequence_ids(window)
                word_ids = encoding.word_ids(window)
                inputs = {
                    "input_ids": encoding["input_ids"][window].unsqueeze(0).to(model_device),
                    "attention_mask": encoding["attention_mask"][window].unsqueeze(0).to(model_device),
                    "token_type_ids": encoding["token_type_ids"][window].unsqueeze(0).to(model_device),
                    "bbox": torch.tensor(_window_bboxes(encoding, window, grid_boxes, sep_id))
                    .unsqueeze(0)
                    .to(model_device),
                }
                with torch.inference_mode():
                    outputs = model(**inputs)
                # Only document tokens may start or end an answer (the pipeline's p_mask).
                allowed = torch.tensor([sid == 1 for sid in sequence_ids], device=model_device)
                start = outputs.start_logits[0].float().masked_fill(~allowed, float("-inf")).softmax(-1)
                end = outputs.end_logits[0].float().masked_fill(~allowed, float("-inf")).softmax(-1)
                candidates = start[:, None] * end[None, :]
                candidates = torch.triu(candidates) - torch.triu(candidates, diagonal=MAX_ANSWER_TOKENS)
                flat = int(candidates.argmax())
                s_index, e_index = divmod(flat, candidates.shape[1])
                score = float(candidates[s_index, e_index])
                if score > best["score"] and word_ids[s_index] is not None and word_ids[e_index] is not None:
                    best = {
                        "start": word_ids[s_index],
                        "end": word_ids[e_index],
                        "score": score,
                        "n_windows": n_windows,
                    }
            return best

        return cls(
            runner, resolved_device, "float32", source, count_windows, _model=model, _tokenizer=tokenizer
        )

    def answer(
        self,
        question: str,
        *,
        words: Sequence[str],
        boxes: Sequence[Sequence[float]],
        image_size: Sequence[int],
    ) -> dict[str, Any]:
        """Answer one question from the page's words and boxes; ``answer`` is the selected word span."""
        checked_words, grid, size = _check_document(words, boxes, image_size)
        checked_question = _check_question(question)
        raw = self._runner(checked_question, checked_words, grid)
        if not isinstance(raw, dict) or "start" not in raw or "end" not in raw or "score" not in raw:
            raise RuntimeError("runner must return a dict with 'start', 'end' and 'score'")
        start, end = raw["start"], raw["end"]
        if start is not None and not (0 <= int(start) <= int(end) < len(checked_words)):
            raise RuntimeError(f"runner returned an invalid word span {start}..{end}")
        span_words = checked_words[start : end + 1] if start is not None else []
        return {
            "answer": " ".join(span_words),
            "score": float(raw["score"]),
            "start": None if start is None else int(start),
            "end": None if end is None else int(end),
            "question": checked_question,
            "n_words": len(checked_words),
            "n_windows": int(raw.get("n_windows", 1)),
            "image_size": list(size),
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation contract ---------------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._tokenizer is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._tokenizer

    def check_fit(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Split validated records into those whose question + page fit one 512-token window and those that
        would be windowed at inference (a windowed page is answerable but is not trained on; nothing is
        truncated)."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if self._count_windows is None:
            raise ValueError("check_fit needs a pipeline built with from_pretrained() or from_artifact()")
        checked = validate_dataset(records, min_records=1, max_records=MAX_RECORDS_FIT)["records"]
        fitting, dropped = [], []
        for record in checked:
            if self._count_windows(record["question"], list(record["words"])) == 1:
                fitting.append(record)
            else:
                dropped.append(record["id"])
        return {"fitting": fitting, "dropped": dropped, "n_fitting": len(fitting), "n_dropped": len(dropped)}

    def evaluate(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Answer every record and score the predictions against its accepted answers (mean ANLS, exact-match
        rate)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import docqa_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        predictions = [
            self.answer(r["question"], words=r["words"], boxes=r["boxes"], image_size=r["image_size"])[
                "answer"
            ]
            for r in checked
        ]
        metrics = docqa_metrics(predictions, [[str(a) for a in r["answers"]] for r in checked])
        metrics.update(
            {
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def _trainable_names(self, trainable_encoder_layers: int) -> list[str]:
        if (
            isinstance(trainable_encoder_layers, bool)
            or not isinstance(trainable_encoder_layers, int)
            or not 1 <= trainable_encoder_layers <= ENCODER_LAYERS
        ):
            raise ValueError(f"trainable_encoder_layers must be an int in 1..{ENCODER_LAYERS}")
        model, _ = self._require_model()
        first = ENCODER_LAYERS - trainable_encoder_layers
        prefixes = tuple(f"layoutlm.encoder.layer.{k}." for k in range(first, ENCODER_LAYERS)) + (
            "qa_outputs.",
        )
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    @staticmethod
    def _span_positions(encoded: Any, index: int, record: Mapping[str, Any]) -> tuple[int, int]:
        """Token start/end of the gold word span inside the page segment of one encoded example."""
        start = end = None
        for pos, (sid, word_id) in enumerate(
            zip(encoded.sequence_ids(index), encoded.word_ids(index), strict=True)
        ):
            if sid != 1:
                continue
            if start is None and word_id == record["answer_start"]:
                start = pos
            if word_id == record["answer_end"]:
                end = pos
        if start is None or end is None or end < start:
            raise ValueError(f"record {record['id']}: gold span does not map onto the encoded page tokens")
        return start, end

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 6,
        lr: float = 3e-5,
        batch_size: int = 16,
        trainable_encoder_layers: int = DEFAULT_TRAINABLE_ENCODER_LAYERS,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded supervised fine-tuning on a validated document-QA dataset.

        Only the last `trainable_encoder_layers` encoder blocks and the span head train (4 blocks by default:
        28,353,026 of 127,792,898 parameters; the word, position and 2-D box embeddings and the earlier blocks
        stay frozen). Start/end cross-entropy on the gold word span's first and last tokens, AdamW at a fixed
        learning rate with gradient clipping at 1.0, dynamic padding, no scheduler; every training record must
        fit one window (`check_fit`). Epoch 0 records the frozen model's validation ANLS; the epoch with the
        highest validation ANLS is kept."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        names = self._trainable_names(trainable_encoder_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        if self._count_windows is not None:
            over = [
                r["id"] for r in train_checked if self._count_windows(r["question"], list(r["words"])) != 1
            ]
            if over:
                raise ValueError(
                    f"{len(over)} training record(s) need more than one window (use check_fit): {over[:5]}"
                )
        import torch

        torch.manual_seed(seed)
        model, tokenizer = self._require_model()
        sep_id = tokenizer.sep_token_id
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        device = next(model.parameters()).device

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            return {k: v for k, v in self.evaluate(val_checked).items() if k in ("anls", "exact_match", "n")}

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_anls = entry["val"]["anls"] if entry["val"] else -math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        initial_state = {k: v.clone() for k, v in best_state.items()}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                order = torch.randperm(len(train_checked), generator=generator).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    batch = [train_checked[i] for i in order[start : start + batch_size]]
                    encoded = tokenizer(
                        [r["question"].split() for r in batch],
                        [list(r["words"]) for r in batch],
                        is_split_into_words=True,
                        padding=True,
                        truncation="only_second",
                        max_length=MAX_SEQ_LEN,
                        return_token_type_ids=True,
                        return_tensors="pt",
                    )
                    grids = [
                        [normalize_box(box, r["image_size"][0], r["image_size"][1]) for box in r["boxes"]]
                        for r in batch
                    ]
                    bbox = torch.tensor(
                        [_window_bboxes(encoded, i, grids[i], sep_id) for i in range(len(batch))]
                    )
                    positions = [self._span_positions(encoded, i, r) for i, r in enumerate(batch)]
                    out = model(
                        input_ids=encoded["input_ids"].to(device),
                        attention_mask=encoded["attention_mask"].to(device),
                        token_type_ids=encoded["token_type_ids"].to(device),
                        bbox=bbox.to(device),
                        start_positions=torch.tensor([s for s, _e in positions], device=device),
                        end_positions=torch.tensor([e for _s, e in positions], device=device),
                    )
                    optimiser.zero_grad(set_to_none=True)
                    out.loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimiser.step()
                    losses.append(float(out.loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
                history.append(entry)
                if progress:
                    progress(entry)
                current = entry["val"]["anls"] if entry["val"] else math.inf
                if current > best_anls or not entry["val"]:
                    best_anls = current
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in training, validation or the progress callback leaves the base
            # exactly as it was, with every parameter frozen again.
            restore = dict(model.state_dict())
            restore.update(initial_state)
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_encoder_layers": trainable_encoder_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation ANLS" if val_checked else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted encoder-block and span-head tensors as safetensors plus a base manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> Path:
        """Refuse an artifact whose manifest is not exactly the one this pipeline writes: the supported format
        and version, the pinned base (id, revision, weight file, digest), exactly one file entry named
        `adapter.safetensors` that resolves inside the artifact directory, and a recorded
        `trainable_encoder_layers` in range. Nothing is deserialised here. The digest check that follows
        detects corruption or drift of the weights relative to the adjacent manifest; it is not authenticity
        against an actor who can replace both files."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not the supported "
                f"{ARTIFACT_FORMAT_VERSION!r}"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        if base.get("weight_file", WEIGHT_FILE) != WEIGHT_FILE:
            raise ValueError("artifact was adapted from a different base weight file")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weight path must resolve inside the artifact directory")
        adapter = manifest.get("adapter")
        layers = adapter.get("trainable_encoder_layers") if isinstance(adapter, Mapping) else None
        if isinstance(layers, bool) or not isinstance(layers, int) or not 1 <= layers <= ENCODER_LAYERS:
            raise ValueError("artifact manifest does not record an in-range integer trainable_encoder_layers")
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite
        exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path = self._check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        # The exact tensor set the recorded configuration implies — no subset, no extra, no other layer.
        expected = sorted(self._trainable_names(manifest["adapter"]["trainable_encoder_layers"]))
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor list does not match its recorded configuration")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not (
                key.startswith("layoutlm.encoder.layer.") or key.startswith("qa_outputs.")
            ):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable encoder or span-head tensor of the base"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key} has shape {tuple(value.shape)}, "
                    f"base has {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> LayoutLMDocumentQAPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 2/3:** `src/layoutlm_document_qa_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Corpus-level document-QA metrics and two non-neural baselines.

The per-question helpers (`normalize_answer`, `anls`, `exact_match`) live in `pipeline.py` and follow DocVQA's
conventions. This module averages them over a dataset and adds two baselines a fine-tuned reader must beat:
**last number** (the last word on the page that contains a digit — receipts end with their totals) and
**keyword lookup** (find the page word that best matches a content word of the question, e.g. `TOTAL` for
"What is the total amount?", and answer with the next word after it that contains a digit — a lookup with no
model that stands in for the "find the label, read the value" heuristic a person applies to a receipt).
"""

from __future__ import annotations

import re
from collections.abc import Mapping, Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import anls, exact_match, normalize_answer` removed — names are kernel globals defined by the carried modules

_DIGIT_RE = re.compile(r"\d")
# Function words removed from a question before its remaining words are matched against the page.
QUESTION_STOP_WORDS = frozenset(
    [
        "what", "is", "the", "how", "much", "many", "was", "were", "paid", "given", "by", "of", "name",
        "first", "item", "items", "bought", "amount", "charge",
    ]
)  # fmt: skip
METRIC_DEFINITIONS = {
    "anls": (
        "mean over questions of the Average Normalised Levenshtein Similarity between the prediction and its "
        "best-matching accepted answer (lower-cased, punctuation removed, whitespace collapsed; a similarity "
        "below 0.5 scores 0); in 0..1"
    ),
    "exact_match": (
        "fraction of questions whose normalised prediction equals a normalised accepted answer; in 0..1"
    ),
}


def docqa_metrics(predictions: Sequence[str], golds: Sequence[Sequence[str]]) -> dict[str, Any]:
    """Mean ANLS and exact-match rate over parallel predictions and accepted-answer lists."""
    if len(predictions) != len(golds):
        raise ValueError(f"{len(predictions)} predictions but {len(golds)} gold lists")
    if not predictions:
        raise ValueError("no predictions to score")
    scores = [anls(p, g) for p, g in zip(predictions, golds, strict=True)]
    exact = [exact_match(p, g) for p, g in zip(predictions, golds, strict=True)]
    return {
        "n": len(predictions),
        "anls": sum(scores) / len(scores),
        "exact_match": sum(exact) / len(exact),
        "empty_rate": sum(1 for p in predictions if not p.strip()) / len(predictions),
        "definitions": dict(METRIC_DEFINITIONS),
    }


def _golds(records: Sequence[Mapping[str, Any]]) -> list[list[str]]:
    return [[str(a) for a in r["answers"]] for r in records]


def last_number_answer(words: Sequence[str]) -> str:
    """The last word on the page containing a digit; empty when there is none."""
    for word in reversed(list(words)):
        if _DIGIT_RE.search(word):
            return word
    return ""


def last_number_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Answer every question with the last numeric word of its page."""
    result = docqa_metrics([last_number_answer(r["words"]) for r in records], _golds(records))
    result["baseline"] = "last word on the page containing a digit"
    return result


def _overlap(a: str, b: str) -> int:
    """Length of the longest common prefix of two normalised words (`subtotal` vs `subtot` → 6)."""
    n = 0
    for x, y in zip(a, b, strict=False):
        if x != y:
            break
        n += 1
    return n


def keyword_lookup_answer(question: str, words: Sequence[str], *, min_overlap: int = 3) -> str:
    """Find the page word sharing the longest prefix (at least `min_overlap` characters, last such word on
    ties) with any content word of the question; answer with the first word after it that contains a digit,
    or the last numeric word of the page when no keyword matches."""
    content = [w for w in normalize_answer(question).split() if w not in QUESTION_STOP_WORDS]
    page = [normalize_answer(w) for w in words]
    best_index, best_overlap = None, min_overlap - 1
    for index, word in enumerate(page):
        overlap = max((_overlap(word, q) for q in content), default=0)
        if overlap >= best_overlap and overlap >= min_overlap:
            best_index, best_overlap = index, overlap
    if best_index is not None:
        for word in list(words)[best_index + 1 :]:
            if _DIGIT_RE.search(word):
                return word
    return last_number_answer(words)


def keyword_lookup_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """A label-then-value lookup with no model."""
    predictions = [keyword_lookup_answer(r["question"], r["words"]) for r in records]
    result = docqa_metrics(predictions, _golds(records))
    result["baseline"] = "keyword lookup (best-matching page word, then the next numeric word)"
    return result

**Module 3/3:** `src/layoutlm_document_qa_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Document-QA dataset contract for fine-tuning: the pinned CORD-v2 receipt sample, validation, seeded
page-disjoint splitting, BYOD loaders and JSONL export.

The default dataset is **real**: CORD-v2 (Park et al., 2019; NAVER Clova; CC BY 4.0) — photographed Indonesian
receipts whose every printed word carries a pixel box and a field category (`total.total_price`,
`sub_total.tax_price`, `menu.nm`, …). LayoutLM reads words and boxes, never pixels, so this module fetches
**only the `ground_truth` column** of the two pinned Hub parquet files (the 100-receipt `test` and
`validation` shards): the parquet footer and the one column chunk are read over HTTP range requests through
`pyarrow` (about 0.4 MB of the 476 MB the two files hold with their images). Each file is pinned three
ways — the immutable dataset revision in the URL, the byte size and SHA-256 the Hub declares for the file
(checked against the file metadata before any byte is read), and a SHA-256 of the decoded column (checked
after reading) — and refused on any mismatch. Questions are templated from the field categories: a category
that occurs on exactly one line of a receipt becomes one question whose gold answer is that line's value
words, so every answer is a contiguous span of the page's OCR words by construction.

A record is ``{id, page_id, question, words, boxes, image_size, answer_start, answer_end, answers}`` — the
page's words with one pixel xyxy box each and the inclusive word indices of the gold span; ``answers`` holds
the span text. Every question on the same page lands in the same split.
"""

from __future__ import annotations

import hashlib
import io
import json
import random
import re
import urllib.request
from collections.abc import Callable, Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_QUESTION_CHARS, MODEL_ID, _check_document, _check_question` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "CORD-v2"
CORPUS_REPO = "naver-clova-ix/cord-v2"
CORPUS_REVISION = "7f0115a4b758a71d6473b8d085751692da2fef98"
CORPUS_RELEASE = "Hugging Face Hub dataset revision 7f0115a4 (2022-07-19)"
CORPUS_LICENSE = "CC BY 4.0 (Park et al. 2019; NAVER Clova; huggingface.co/datasets/naver-clova-ix/cord-v2)"
CORPUS_COLUMN = "ground_truth"
# The two shards read: path, size and SHA-256 as the Hub declares them for the LFS object (the whole file is
# never downloaded, so these are checked against the file metadata), the row count, and the SHA-256 of the
# decoded `ground_truth` column (canonical JSON list of the row strings), checked after the read.
CORPUS_FILES: dict[str, dict[str, Any]] = {
    "test": {
        "path": "data/test-00000-of-00001-9c204eb3f4e11791.parquet",
        "bytes": 234_202_795,
        "sha256": "51c65f1788faff392abe2a0b55b023eb23e9be551c509138eaa3a832514224e7",
        "rows": 100,
        "column_sha256": "b499e58aa298e5242b5222b7e333e92e6affb79a192def149d0a0aac3e41d9f1",
    },
    "validation": {
        "path": "data/validation-00000-of-00001-cc3c5779fe22e8ca.parquet",
        "bytes": 242_080_800,
        "sha256": "0d0f6dac11fdcc549de2746aa9f53136a3bc22a2a1aff2b0b847f7622ad60c15",
        "rows": 100,
        "column_sha256": "adf8303ec0295af1ef11a63dd0b72453e95399cfd2cd76e3d494c284730bdef9",
    },
}
DEFAULT_CACHE_DIR = Path("weights") / "cord-v2"
# One question per field category that occurs on exactly one line of the receipt; the gold answer is that
# line's words. `menu.nm` (item names) occurs on most receipts several times, so it asks for the *first*
# item — the topmost `menu.nm` line — and is skipped when that item's name spans more than one line.
QUESTION_TEMPLATES: dict[str, str] = {
    "total.total_price": "What is the total amount?",
    "sub_total.subtotal_price": "What is the subtotal?",
    "sub_total.tax_price": "What is the tax amount?",
    "sub_total.service_price": "What is the service charge?",
    "sub_total.discount_price": "What is the discount amount?",
    "total.cashprice": "How much cash was paid?",
    "total.changeprice": "How much change was given?",
    "total.creditcardprice": "How much was paid by card?",
    "total.menuqty_cnt": "How many items were bought?",
    "menu.nm": "What is the name of the first item?",
}
FIRST_ITEM_CATEGORY = "menu.nm"
SAMPLE_SEED = 42
SAMPLE_SPLIT = {
    "train": 119,
    "validation": 30,
    "test": 50,
}  # receipts (pages), not questions; 199 unique pages
MIN_RECORDS = 8
MAX_RECORDS = 20_000
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def column_digest(rows: Sequence[str]) -> str:
    """SHA-256 of the decoded column as a canonical JSON list of its row strings."""
    return _sha256_bytes(json.dumps(list(rows), ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


class _HttpRangeFile(io.RawIOBase):
    """A seekable read-only view of one HTTPS object served with `Range` requests (what `pyarrow` needs to
    read a parquet footer and a single column chunk without downloading the file)."""

    def __init__(self, url: str, size: int) -> None:
        self.url, self.size, self.pos = url, size, 0
        self.fetched = 0

    def readable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return True

    def tell(self) -> int:
        return self.pos

    def seek(self, offset: int, whence: int = 0) -> int:
        base = {0: 0, 1: self.pos, 2: self.size}[whence]
        self.pos = max(0, base + offset)
        return self.pos

    def read(self, n: int = -1) -> bytes:
        if n is None or n < 0:
            n = self.size - self.pos
        if n <= 0 or self.pos >= self.size:
            return b""
        end = min(self.size, self.pos + n) - 1
        request = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{end}"})
        with urllib.request.urlopen(request, timeout=180) as response:  # noqa: S310 (pinned https URL)
            if response.status != 206:
                raise ValueError(f"{self.url}: server ignored the Range request (HTTP {response.status})")
            data = response.read()
        self.fetched += len(data)
        self.pos += len(data)
        return data

    def readinto(self, buffer: Any) -> int:
        data = self.read(len(buffer))
        buffer[: len(data)] = data
        return len(data)


def _hub_column(split: str, spec: Mapping[str, Any]) -> list[str]:
    """Read the pinned shard's `ground_truth` column from the Hub: the file's declared size and LFS SHA-256
    are checked against the pins first, then only the parquet footer and that column are fetched."""
    import pyarrow.parquet as pq
    from huggingface_hub import get_hf_file_metadata, hf_hub_url

    url = hf_hub_url(CORPUS_REPO, spec["path"], repo_type="dataset", revision=CORPUS_REVISION)
    metadata = get_hf_file_metadata(url)
    declared = (metadata.etag or "").strip('"')
    if metadata.size != spec["bytes"] or declared != spec["sha256"]:
        raise ValueError(
            f"{split} shard: the Hub declares {metadata.size} bytes / sha256 {declared[:16]}…, "
            f"pinned {spec['bytes']} / {spec['sha256'][:16]}…"
        )
    handle = _HttpRangeFile(url, spec["bytes"])
    table = pq.ParquetFile(handle).read(columns=[CORPUS_COLUMN])
    return [str(value) for value in table.column(CORPUS_COLUMN).to_pylist()]


def fetch_corpus(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Callable[[str, Mapping[str, Any]], Sequence[str]] | None = None,
) -> dict[str, list[str]]:
    """Return the pinned shards' `ground_truth` rows per split from the cache or the Hub, digest-verified."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out: dict[str, list[str]] = {}
    for split, spec in CORPUS_FILES.items():
        local = cache / f"{split}.{CORPUS_COLUMN}.json"
        rows: list[str] | None = None
        if local.is_file():
            cached = json.loads(local.read_text(encoding="utf-8"))
            if isinstance(cached, list) and column_digest(cached) == spec["column_sha256"]:
                rows = [str(value) for value in cached]
        if rows is None:
            rows = [
                str(value)
                for value in (fetcher(split, spec) if fetcher is not None else _hub_column(split, spec))
            ]
            if len(rows) != spec["rows"] or column_digest(rows) != spec["column_sha256"]:
                raise ValueError(
                    f"{split} shard: fetched {len(rows)} rows with column sha256 "
                    f"{column_digest(rows)[:16]}…, "
                    f"pinned {spec['rows']} / {spec['column_sha256'][:16]}…"
                )
            local.write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
        out[split] = rows
    return out


def _quad_to_box(quad: Mapping[str, Any], width: int, height: int) -> list[float] | None:
    xs = [float(quad[k]) for k in ("x1", "x2", "x3", "x4")]
    ys = [float(quad[k]) for k in ("y1", "y2", "y3", "y4")]
    x0, y0 = max(0.0, min(xs)), max(0.0, min(ys))
    x1, y1 = min(float(width), max(xs)), min(float(height), max(ys))
    if x1 <= x0 or y1 <= y0:
        return None
    return [x0, y0, x1, y1]


def page_from_ground_truth(ground_truth: str | Mapping[str, Any], page_id: str) -> dict[str, Any]:
    """One CORD `ground_truth` JSON → the page's words, boxes, size and the labelled lines over them.

    Words are concatenated line by line in the annotation's order (`valid_line`), every word box is the
    axis-aligned hull of its quadrilateral clipped to the page, and words with an empty text or a degenerate
    box are dropped. Each line records its field `category`, `group_id`, its inclusive word span and the span
    of its **value** words (`is_key` 0 — a printed key such as `TOTAL` is part of the line but not of the
    answer); `value_start` is `None` when the value words are not one contiguous run."""
    data = json.loads(ground_truth) if isinstance(ground_truth, str) else ground_truth
    size = data["meta"]["image_size"]
    width, height = int(size["width"]), int(size["height"])
    words: list[str] = []
    boxes: list[list[float]] = []
    lines: list[dict[str, Any]] = []
    for line in data.get("valid_line", []):
        start = len(words)
        values: list[int] = []
        for word in line.get("words", []):
            text = " ".join(str(word.get("text", "")).split())
            box = _quad_to_box(word["quad"], width, height) if text else None
            if box is None:
                continue
            if not int(word.get("is_key", 0)):
                values.append(len(words))
            words.append(text)
            boxes.append(box)
        if len(words) > start:
            contiguous = bool(values) and values == list(range(values[0], values[-1] + 1))
            lines.append(
                {
                    "category": str(line.get("category", "")),
                    "group_id": int(line.get("group_id", -1)),
                    "start": start,
                    "end": len(words) - 1,
                    "value_start": values[0] if contiguous else None,
                    "value_end": values[-1] if contiguous else None,
                }
            )
    return {"id": page_id, "words": words, "boxes": boxes, "image_size": [width, height], "lines": lines}


def read_corpus(rows: Mapping[str, Sequence[str]]) -> dict[str, list[dict[str, Any]]]:
    """The fetched shards as pages, ids `<split>-<row>`."""
    out: dict[str, list[dict[str, Any]]] = {}
    for split, values in rows.items():
        out[split] = [
            page_from_ground_truth(value, f"{split}-{index:03d}") for index, value in enumerate(values)
        ]
    return out


def questions_for_page(page: Mapping[str, Any]) -> list[dict[str, Any]]:
    """The templated questions a page supports: one per category on exactly one line, plus the first item;
    the gold answer is the line's contiguous run of value words (lines without one are skipped)."""
    by_category: dict[str, list[dict[str, Any]]] = {}
    for line in page["lines"]:
        by_category.setdefault(line["category"], []).append(line)
    out = []
    for category, question in QUESTION_TEMPLATES.items():
        candidates = by_category.get(category, [])
        if not candidates:
            continue
        if category == FIRST_ITEM_CATEGORY:
            topmost = min(candidates, key=lambda line: (page["boxes"][line["start"]][1], line["start"]))
            same_group = [line for line in candidates if line["group_id"] == topmost["group_id"]]
            if len(same_group) != 1:
                continue
            line = topmost
        elif len(candidates) == 1:
            line = candidates[0]
        else:
            continue
        if line.get("value_start") is None:
            continue
        start, end = int(line["value_start"]), int(line["value_end"])
        out.append(
            {
                "page_id": page["id"],
                "question": question,
                "field": category,
                "words": list(page["words"]),
                "boxes": [list(box) for box in page["boxes"]],
                "image_size": list(page["image_size"]),
                "answer_start": start,
                "answer_end": end,
                "answers": [" ".join(page["words"][start : end + 1])],
            }
        )
    return out


def build_sample_dataset(
    pages: Mapping[str, Sequence[Mapping[str, Any]]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Pool every fetched page that supports at least one question, drop any page whose lower-cased word
    sequence repeats an earlier page (CORD-v2 holds a few identical receipts across its shards), shuffle the
    pages with `seed`, cut **by page** into `sizes` (train / validation / test receipts) and expand each page
    into its questions — so no page, and no question, is shared between splits."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    pool = [dict(page) for split in sorted(pages) for page in pages[split]]
    seen: set[tuple[str, ...]] = set()
    supported = []
    for page in pool:
        key = _page_key(page)
        if key in seen or not questions_for_page(page):
            continue
        seen.add(key)
        supported.append(page)
    needed = sum(sizes.values())
    if len(supported) < needed:
        raise ValueError(f"{len(supported)} pages support a question; the split sizes need {needed}")
    random.Random(seed).shuffle(supported)
    out: dict[str, list[dict[str, Any]]] = {}
    offset = 0
    for name in ("train", "validation", "test"):
        chosen = supported[offset : offset + sizes[name]]
        offset += sizes[name]
        records = [record for page in chosen for record in questions_for_page(page)]
        out[name] = [{"id": f"{name}-{index:04d}", **record} for index, record in enumerate(records)]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Callable[[str, Mapping[str, Any]], Sequence[str]] | None = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned shards."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(
            f"{label} must be a mapping with id/question/words/boxes/image_size/answer_start/answer_end"
        )
    for key in ("id", "question", "words", "boxes", "image_size", "answer_start", "answer_end"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label}: id must match {_ID_RE.pattern}")
    try:
        words, _grid, size = _check_document(record["words"], record["boxes"], record["image_size"])
        question = _check_question(record["question"])
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{label}: {exc}") from exc
    start, end = record["answer_start"], record["answer_end"]
    if any(isinstance(v, bool) or not isinstance(v, int) for v in (start, end)):
        raise ValueError(f"{label}: answer_start and answer_end must be ints")
    if not 0 <= start <= end < len(words):
        raise ValueError(f"{label}: answer span {start}..{end} is not inside the {len(words)} words")
    text = " ".join(words[start : end + 1])
    answers = record.get("answers", [text])
    if isinstance(answers, str) or not isinstance(answers, Sequence) or not answers or answers[0] != text:
        raise ValueError(f"{label}: answers[0] must be the span text {text[:40]!r}")
    item = {
        "id": rid,
        "page_id": str(record.get("page_id", rid)),
        "question": question,
        "words": words,
        "boxes": [[float(v) for v in box] for box in record["boxes"]],
        "image_size": list(size),
        "answer_start": start,
        "answer_end": end,
        "answers": [str(a) for a in answers],
    }
    if "field" in record:
        item["field"] = str(record["field"])
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a document-QA dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError(
            "records must be a list of {id, question, words, boxes, image_size, answer_start, answer_end}"
        )
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    pages: set[str] = set()
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        pages.add(item["page_id"])
        checked.append(item)
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_pages": len(pages),
        "words_per_page": {
            "min": min(len(r["words"]) for r in checked),
            "max": max(len(r["words"]) for r in checked),
        },
        "answer_words": {
            "min": min(r["answer_end"] - r["answer_start"] + 1 for r in checked),
            "max": max(r["answer_end"] - r["answer_start"] + 1 for r in checked),
        },
        "question_chars": {
            "min": min(len(r["question"]) for r in checked),
            "max": max(len(r["question"]) for r in checked),
        },
        "max_question_chars": MAX_QUESTION_CHARS,
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [
        [r["id"], r["question"], r["words"], r["boxes"], r["image_size"], r["answer_start"], r["answer_end"]]
        for r in records
    ]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def gold_texts(record: Mapping[str, Any]) -> list[str]:
    """The accepted answers of a record (the gold span text)."""
    return [str(a) for a in record["answers"]]


def _page_key(record: Mapping[str, Any]) -> tuple[str, ...]:
    return tuple(str(w).lower() for w in record["words"])


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no page id and no page content (its lower-cased word sequence) appears in two splits."""
    seen_ids: dict[str, str] = {}
    seen_words: dict[tuple[str, ...], str] = {}
    for name, records in splits.items():
        for record in records:
            page_id = str(record.get("page_id", record["id"]))
            if page_id in seen_ids and seen_ids[page_id] != name:
                raise ValueError(f"page {page_id!r} appears in both {seen_ids[page_id]} and {name}")
            seen_ids[page_id] = name
            key = _page_key(record)
            if key in seen_words and seen_words[key] != name:
                raise ValueError(
                    f"a page's words ({' '.join(key[:6])!r}…) appear in both {seen_words[key]} and {name}"
                )
            seen_words[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded split of a BYOD dataset into train/validation/test **by page**: every question on the same page
    lands in the same split, so a test page is never seen in training."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    groups: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        groups.setdefault(record["page_id"], []).append(record)
    order = list(groups.values())
    random.Random(seed).shuffle(order)
    n_test = max(1, round(len(checked) * test_fraction))
    n_val = round(len(checked) * val_fraction)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for group in order:
        if len(splits["test"]) < n_test:
            splits["test"].extend(group)
        elif len(splits["validation"]) < n_val:
            splits["validation"].extend(group)
        else:
            splits["train"].extend(group)
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read records from a JSON array or a JSONL file of
    ``{id, page_id, question, words, boxes, image_size, answer_start, answer_end}`` objects."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix == ".json":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON dataset must be an array of records")
        return data
    raise ValueError("BYOD datasets must be .json or .jsonl")


def write_dataset_jsonl(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """One record per line in the shape `load_byod_dataset` reads back."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    keys = (
        "id",
        "page_id",
        "question",
        "words",
        "boxes",
        "image_size",
        "answer_start",
        "answer_end",
        "answers",
    )
    with open(out, "w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps({k: record[k] for k in keys if k in record}, ensure_ascii=False) + "\n")
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `beed3c4d02d8…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `LayoutLMDocumentQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "layoutlm-document-qa",
  "modelId": "impira/layoutlm-document-qa",
  "revision": "beed3c4d02d86017ebca5bd0fdf210046b907aa6",
  "files": [
    {
      "path": "README.md",
      "bytes": 2326,
      "sha256": "40fd65fc734cc7eb73cc815465b8073bc4e5b7484406e537c7b8d763f88493f7"
    },
    {
      "path": "config.json",
      "bytes": 789,
      "sha256": "6d0fc068193109d0d053fa4de00963778beffbde05067c3e9f3454235044380f"
    },
    {
      "path": "merges.txt",
      "bytes": 456356,
      "sha256": "fe36cab26d4f4421ed725e10a2e9ddb7f799449c603a96e7f29b5a3c82a95862"
    },
    {
      "path": "model.safetensors",
      "bytes": 511200628,
      "sha256": "e4bbad3e4a1b5ae50c787b7afd6049a0bfa99fd823b50436e444e092ae2347b9"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 239,
      "sha256": "378eb3bf733eb16e65792d7e3fda5b8a4631387ca04d2015199c4d4f22ae554d"
    },
    {
      "path": "tokenizer.json",
      "bytes": 1355881,
      "sha256": "33465117406b9007673e8ba283f7f1383d9b5094df947481af60eec94ed7d7bd"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 315,
      "sha256": "ea11996be5d083d63c72700810b18a6cfdf78c55131b754a75ae94e66b0ad6ab"
    },
    {
      "path": "vocab.json",
      "bytes": 798293,
      "sha256": "ed19656ea1707df69134c4af35c8ceda2cc9860bf2c3495026153a133670ab5e"
    }
  ],
  "totalBytes": 513814827
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = LayoutLMDocumentQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. CORD-v2 receipt corpus, validation and split

`fetch_corpus` reads the pinned shards' `ground_truth` column (or the cache under `weights/cord-v2/`): for each shard it first checks the byte size and SHA-256 the Hub declares for the file against the pins, then reads only the parquet footer and the one column chunk through `pyarrow` over HTTPS range requests, and refuses the decoded column unless its SHA-256 matches. `read_corpus` turns each row into a page — the words of every annotated line in order, each box the clipped axis-aligned hull of the annotated quadrilateral, and the lines' field categories with their value-word spans (`is_key` 0; a printed key such as `TOTAL` is on the line but not in the answer). `build_sample_dataset` keeps every receipt that supports at least one templated question, drops the one receipt whose words repeat another shard's, shuffles the 199 unique receipts with `SPLIT_SEED` and cuts them **by receipt** into 119 / 30 / 50 training, validation and test pages, then expands each page into its questions (a category that occurs on exactly one line becomes one question; *first item* is the topmost single-line `menu.nm`). `validate_dataset` checks every record against the contract, `check_split_disjoint` asserts no page id and no page content is shared, and the training split is written to `outputs/layoutlm_document_qa_train.jsonl` in the shape BYOD expects.

Look for: 100 + 100 raw rows, three digests, splits 595 / 152 / 229 questions over 119 / 30 / 50 receipts, the field mix, and four refusal probes — a duplicate id, a gold span outside the words, a box outside the page and a dataset too small to split — each rejected before `torch` does anything.

In [ ]:
import collections
import hashlib
import io
import json

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    corpus_rows = fetch_corpus(cache_dir='weights/cord-v2')
    raw_rows = {name: len(rows) for name, rows in corpus_rows.items()}
    splits = build_sample_dataset(read_corpus(corpus_rows), seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} {CORPUS_RELEASE} ({CORPUS_LICENSE})'
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
disjoint = check_split_disjoint(splits)
pages = {name: len({r['page_id'] for r in part}) for name, part in splits.items()}
fields = {name: dict(collections.Counter(r.get('field', 'byod') for r in part)) for name, part in splits.items()}
write_dataset_jsonl(splits['train'], 'outputs/layoutlm_document_qa_train.jsonl')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'pages': pages, 'column_sha256': {name: spec['column_sha256'][:16] + '...' for name, spec in CORPUS_FILES.items()}})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'unique_pages': manifest['unique_pages'], 'words_per_page': manifest['words_per_page'], 'answer_words': manifest['answer_words'], 'digest': manifest['digest'][:16] + '...'}})
print({'fields_test': fields['test']})
example = splits['train'][0]
print({'example': {'id': example['id'], 'page_id': example['page_id'], 'question': example['question'], 'answers': example['answers'], 'span': [example['answer_start'], example['answer_end']], 'words': ' '.join(example['words'][:18]) + ' ...'}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in splits['train'][:8]],
    'gold span outside the words': [{**splits['train'][0], 'answer_end': len(splits['train'][0]['words'])}, *splits['train'][1:8]],
    'box outside the page': [{**splits['train'][0], 'boxes': [[0, 0, splits['train'][0]['image_size'][0] + 1, 5], *splits['train'][0]['boxes'][1:]]}, *splits['train'][1:8]],
    'too small': splits['train'][:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Fit check, then answer through the inference contract

The window ceiling needs the real tokenizer, so it is applied now that the model is loaded: `pipe.check_fit` partitions each split into the records whose question and page fit one 512-token window and the ones that would be **windowed at inference and are therefore not trained on** — nothing is truncated. The dropped ids are printed and the fitting records are what every later cell uses (the build record dropped none of the 976 sample records; receipts are short).

Then the inference contract is exercised as the inference-only tutorial exercised it: an invoice-style form is rendered in code with Pillow's bundled font and the renderer records the pixel box of **every word it draws** — perfect OCR by construction, a different document family from the receipts, and a page the model will be asked to read again after adaptation. `validate_inputs` applies exactly the checks `answer` applies (page size, 1..`MAX_WORDS` non-empty words with one box each inside the page, questions up to `MAX_QUESTION_CHARS`) and returns an input manifest; a box outside the page is validated too and its rejection recorded as a finding. `answer` returns the word span, its inclusive `start`/`end` indices (so the answer's boxes are known), `score`, `n_words`, `n_windows` and the model identity. **Score semantics:** the `score` is the **product of the start and end softmax probabilities of the chosen span within its window** — a ranking signal over spans of this page, **not a calibrated probability** that the answer is right, and never a signal that the question is answerable. Whether the answers are *right* is what Section 6 measures on 229 gold spans, not what five authored pairs can tell you.

In [ ]:
import time

import numpy as np
from PIL import Image, ImageDraw, ImageFont

fit = {name: pipe.check_fit(part) for name, part in splits.items()}
train_records, val_records, test_records = fit['train']['fitting'], fit['validation']['fitting'], fit['test']['fitting']
print({'fit_check': {name: {'fitting': f['n_fitting'], 'dropped': f['dropped']} for name, f in fit.items()}})
ceilings = {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_WORDS': MAX_WORDS, 'MAX_QUESTION_CHARS': MAX_QUESTION_CHARS, 'MAX_SEQ_LEN': MAX_SEQ_LEN, 'DOC_STRIDE': DOC_STRIDE, 'MAX_ANSWER_TOKENS': MAX_ANSWER_TOKENS, 'BOX_GRID': BOX_GRID, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS}
print(ceilings)


def synthetic_form(width=850, height=1100):
    """Invoice-style form rendered with Pillow's bundled font; every drawn word is recorded with its pixel box."""
    page = Image.new('RGB', (width, height), 'white')
    d = ImageDraw.Draw(page)
    body, bold, head = ImageFont.load_default(size=18), ImageFont.load_default(size=20), ImageFont.load_default(size=30)
    words, boxes = [], []

    def put(x, y, text, font, fill='black'):
        for token in text.split():
            x0, y0, x1, y1 = d.textbbox((x, y), token, font=font)
            d.text((x, y), token, fill=fill, font=font)
            words.append(token)
            boxes.append([float(x0), float(y0), float(x1), float(y1)])
            x = x1 + d.textlength(' ', font=font)

    put(70, 60, 'INVOICE', head)
    put(70, 110, 'Northwind Traders Ltd.', bold)
    put(70, 136, '14 Harbour Road, Portsmouth PO1 3AX', body, (40, 40, 40))
    y = 200
    for label, value in [('Invoice number:', 'NW-2026-0417'), ('Invoice date:', '12 March 2026'), ('Due date:', '11 April 2026'), ('Customer:', 'Blue Yonder Airlines'), ('Purchase order:', 'PO-88213')]:
        put(70, y, label, bold)
        put(300, y, value, body)
        y += 32
    d.rectangle([70, 400, 780, 640], outline='black', width=2)
    cols = [70, 420, 540, 660, 780]
    rows = [('Description', 'Qty', 'Unit price', 'Amount'), ('Cargo pallets (standard)', '40', '$18.50', '$740.00'), ('Shrink wrap rolls', '12', '$9.25', '$111.00'), ('Handling fee', '1', '$65.00', '$65.00')]
    for r, row in enumerate(rows):
        yy = 400 + r * 48
        if r:
            d.line([(70, yy), (780, yy)], fill=(120, 120, 120), width=1)
        for c, cell in enumerate(row):
            put(cols[c] + 10, yy + 14, cell, bold if r == 0 else body)
    for c in cols[1:-1]:
        d.line([(c, 400), (c, 640)], fill=(120, 120, 120), width=1)
    put(540, 670, 'Subtotal:', bold)
    put(680, 670, '$916.00', body)
    put(540, 700, 'VAT (20%):', bold)
    put(680, 700, '$183.20', body)
    put(540, 736, 'Total due:', head)
    put(680, 736, '$1,099.20', head)
    put(70, 900, 'Payment terms: 30 days from invoice date. Bank: Solent Mutual, sort code 40-11-22.', body, (40, 40, 40))
    qa = [
        ('What is the invoice number?', ['NW-2026-0417']),
        ('Who is the customer?', ['Blue Yonder Airlines']),
        ('What is the total due?', ['$1,099.20', '1,099.20']),
        ('What is the due date?', ['11 April 2026']),
        ('How many cargo pallets were invoiced?', ['40']),
    ]
    return page, words, boxes, qa


image, words, boxes, qa = synthetic_form()
questions, golds = [q for q, _ in qa], [g for _, g in qa]
image_name = 'synthetic_invoice_850x1100.png'
image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
input_manifest = validate_inputs(words, boxes, questions, image_size=image.size, names=[image_name])
try:
    validate_inputs(words[:1], [[0, 0, image.width + 1, 10]], questions, image_size=image.size)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'box-outside-page-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/layoutlm_document_qa_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'page': image_name, 'size': image.size, 'rgb_sha256': image_sha256[:16] + '...', 'n_words': len(words), 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})
results = []
for question in questions:
    started = time.perf_counter()
    result = pipe.answer(question, words=words, boxes=boxes, image_size=image.size)
    results.append({'seconds': round(time.perf_counter() - started, 3), **result})
    print(f"Q: {result['question']}\n   A: {result['answer']!r}  score {result['score']:.4f}  words {result['start']}..{result['end']}  windows {result['n_windows']}")
checks = {
    'one_result_per_question': len(results) == len(questions),
    'span_is_the_indexed_words': all(r['answer'] == ' '.join(words[r['start'] : r['end'] + 1]) for r in results if r['start'] is not None),
    'indices_inside_the_page': all(0 <= r['start'] <= r['end'] < r['n_words'] for r in results if r['start'] is not None),
    'scores_in_unit_interval': all(0.0 <= r['score'] <= 1.0 for r in results),
    'one_window': all(r['n_windows'] == 1 for r in results),
}
if not all(checks.values()):
    raise RuntimeError(f'answer output failed a sanity check: {checks}')
frozen_form = evaluation_report(results, golds, sample_kind='synthetic')
print({'checks': checks, 'frozen_invoice': {m['id']: round(m['value'], 3) for m in frozen_form['metrics']}, 'verdict': frozen_form['verdict']})

## 6. Baselines and the frozen model's score on the test receipts

Three numbers frame the adaptation. The **last-number baseline** answers every question with the last word on the page that contains a digit — receipts end with their totals, so it is right more often than chance and wrong for everything else. The **keyword-lookup baseline** finds the page word sharing the longest prefix with a content word of the question (`TOTAL` for *What is the total amount?*) and answers with the next numeric word after it — the find-the-label-then-read-the-value heuristic a person applies, with no model. The **frozen model** answers the 229 test questions and is scored with the same two metrics: mean **ANLS** (DocVQA's normalised Levenshtein similarity, 0 below 0.5) and the **exact-match** rate, both after lower-casing, punctuation removal and whitespace collapsing. Expect the frozen model to be strong already — it was fine-tuned on DocVQA, and receipt totals are what it reads best — and read the per-field breakdown: the build record measured ANLS 0.98 on subtotals but 0.17 on *How many items were bought?* and 0.79 on item names, which is where Section 7 has room to move.

In [ ]:
baseline_last = last_number_baseline(test_records)
baseline_lookup = keyword_lookup_baseline(test_records)
print({'last_number_baseline': {'anls': round(baseline_last['anls'], 3), 'exact_match': round(baseline_last['exact_match'], 3), 'n': baseline_last['n']}})
print({'keyword_lookup_baseline': {'anls': round(baseline_lookup['anls'], 3), 'exact_match': round(baseline_lookup['exact_match'], 3), 'n': baseline_lookup['n']}})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records)
print({'frozen_model_test': {'anls': round(frozen_test['anls'], 3), 'exact_match': round(frozen_test['exact_match'], 3), 'n': frozen_test['n'], 'verdict': frozen_test['verdict']}, 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_test['definitions']})


def by_field(pipeline, records):
    scores = collections.defaultdict(list)
    for record in records:
        item = pipeline.answer(record['question'], words=record['words'], boxes=record['boxes'], image_size=record['image_size'])
        scores[record.get('field', 'byod')].append(anls(item['answer'], gold_texts(record)))
    return {field: {'n': len(values), 'anls': round(sum(values) / len(values), 2)} for field, values in sorted(scores.items())}


frozen_fields = by_field(pipe, test_records)
print({'frozen_by_field': frozen_fields})
for record in test_records[:3]:
    item = pipe.answer(record['question'], words=record['words'], boxes=record['boxes'], image_size=record['image_size'])
    print({'question': record['question'], 'frozen': item['answer'], 'gold': gold_texts(record)})
assert frozen_test['anls'] > baseline_last['anls']

## 7. Bounded fine-tuning of the last encoder blocks and the span head

`pipe.adapt` trains only the last `TRAINABLE_ENCODER_LAYERS` encoder blocks plus the span head `qa_outputs` — four blocks by default, 28,353,026 of 127,792,898 parameters; the word, position and 2-D box embeddings and the earlier blocks stay frozen — with start/end cross-entropy on the first and last token of the gold word span, AdamW at a fixed learning rate, gradient clipping at 1.0, dynamic padding, seeded shuffling and no scheduler. Nothing is truncated: every training record passed the fit check. Epoch 0 records the frozen model's validation ANLS; every epoch is scored on the validation receipts, and the epoch with the highest validation ANLS is kept.

Watch validation ANLS move from about 0.82 to about 0.96 over six epochs (about two and a half minutes per epoch on CPU, validation scoring included; the build record kept epoch 5). The build record's counter-examples are in the model card — two trainable blocks reach about 0.91 test ANLS in the same six epochs; the default is the configuration that captured most of the gain at the same CPU cost.

In [ ]:
EPOCHS = 6  # @param {type:"integer"}
LEARNING_RATE = 3e-5  # @param {type:"number"}
BATCH_SIZE = 16  # @param {type:"integer"}
TRAINABLE_ENCODER_LAYERS = 4  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row['val_anls'] = round(entry['val']['anls'], 4)
        row['val_exact_match'] = round(entry['val']['exact_match'], 4)
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_encoder_layers=TRAINABLE_ENCODER_LAYERS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test receipts were never used for training or epoch selection, and no test page — by id or by word content — appears in the training or validation splits. The adapted model is scored exactly as the frozen model was in Section 6, the four numbers are put side by side and the per-field breakdown is repeated. Look for an ANLS gain of several points concentrated in the fields the frozen model missed — the cell asserts the adapted test ANLS is above the frozen one — and for the same three questions answered by the adapted model. Two hundred and twenty-nine questions over 50 receipts from one seeded split of one corpus give **no dispersion estimate**; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a gain on CORD receipts with their annotated OCR says nothing about your documents or your OCR until you measure it there.

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
adapted_fields = by_field(pipe, test_records)
comparison = {
    'anls': {'last_number': round(baseline_last['anls'], 3), 'keyword_lookup': round(baseline_lookup['anls'], 3), 'frozen': round(frozen_test['anls'], 3), 'adapted': round(adapted_test['anls'], 3)},
    'exact_match': {'last_number': round(baseline_last['exact_match'], 3), 'keyword_lookup': round(baseline_lookup['exact_match'], 3), 'frozen': round(frozen_test['exact_match'], 3), 'adapted': round(adapted_test['exact_match'], 3)},
    'delta_vs_frozen': {'anls': round(adapted_test['anls'] - frozen_test['anls'], 3), 'exact_match': round(adapted_test['exact_match'] - frozen_test['exact_match'], 3)},
    'by_field': {field: {'n': frozen_fields[field]['n'], 'frozen': frozen_fields[field]['anls'], 'adapted': adapted_fields[field]['anls']} for field in frozen_fields},
}
for metric, row in comparison.items():
    print({metric: row})
for record in test_records[:3]:
    item = pipe.answer(record['question'], words=record['words'], boxes=record['boxes'], image_size=record['image_size'])
    print({'question': record['question'], 'adapted': item['answer'], 'gold': gold_texts(record)})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'pages': pages,
    'fit_check': {name: {'fitting': f['n_fitting'], 'dropped': f['dropped']} for name, f in fit.items()},
    'baselines': {'last_number': baseline_last, 'keyword_lookup': baseline_lookup},
    'frozen_test': frozen_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/layoutlm_document_qa_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['anls'] > frozen_test['anls']
print({'report': 'outputs/layoutlm_document_qa_evaluation_report.json'})

## 9. Re-read the invoice, export the adapter and reload it

The five authored invoice questions from Section 5 are answered again by the adapted model — a page from a different document family than the receipts it was tuned on, so this is a small look at what the adaptation did *outside* its corpus (the build record kept 5/5; a lost answer here is a finding to record, not a failure) — and scored with the per-page `evaluation_report`, whose verdict is `sample-sanity` because five authored pairs on one perfectly-OCR'd page carry no dispersion estimate. The answers are written as CSV and an annotated PNG draws the adapted answer spans' boxes on the page.

`pipe.save_artifact` writes the trained tensors — the last four encoder blocks and the span head, about 113 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `LayoutLMDocumentQAPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor that is not an adaptable encoder-block or span-head tensor, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical answers on eight test receipts (VER4).

In [ ]:
import csv
import shutil

adapted_results = [pipe.answer(question, words=words, boxes=boxes, image_size=image.size) for question in questions]
adapted_form = evaluation_report(adapted_results, golds, sample_kind='synthetic')
for before, after in zip(results, adapted_results, strict=True):
    print({'question': before['question'], 'frozen': before['answer'], 'adapted': after['answer'], 'gold': golds[questions.index(before['question'])]})
print({'invoice_after_adaptation': {m['id']: round(m['value'], 3) for m in adapted_form['metrics']}, 'verdict': adapted_form['verdict'], 'reason': adapted_form['reason'][:80] + '...'})
with open('outputs/layoutlm_document_qa_answers.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'question', 'frozen_answer', 'adapted_answer', 'adapted_score', 'start', 'end', 'gold'])
    for before, after, gold in zip(results, adapted_results, golds, strict=True):
        writer.writerow([image_name, after['question'], before['answer'], after['answer'], f"{after['score']:.6f}", after['start'], after['end'], ' | '.join(gold)])
COLOURS = [(200, 30, 30), (0, 140, 0), (40, 90, 220), (200, 120, 0), (130, 0, 160)]
annotated = image.convert('RGB').copy()
draw = ImageDraw.Draw(annotated)
for index, item in enumerate(adapted_results):
    if item['start'] is None:
        continue
    for box in boxes[item['start'] : item['end'] + 1]:
        draw.rectangle([box[0] - 2, box[1] - 2, box[2] + 2, box[3] + 2], outline=COLOURS[index % len(COLOURS)], width=2)
annotated.save('outputs/layoutlm_document_qa_annotated.png')

artifact_dir = Path('outputs/layoutlm_document_qa_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'layoutlm_document_qa', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = LayoutLMDocumentQAPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [pipe.answer(r['question'], words=r['words'], boxes=r['boxes'], image_size=r['image_size'])['answer'] for r in test_records[:8]]
after = [reloaded.answer(r['question'], words=r['words'], boxes=r['boxes'], image_size=r['image_size'])['answer'] for r in test_records[:8]]
parity = {'identical_answers': sum(a == b for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_answers'] == parity['of']

weight_entry = next(entry for entry in MANIFEST['files'] if entry['path'] == WEIGHT_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'repo': CORPUS_REPO, 'revision': CORPUS_REVISION, 'release': CORPUS_RELEASE, 'license': CORPUS_LICENSE, 'column': CORPUS_COLUMN, 'files': CORPUS_FILES},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'page': {'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'n_words': len(words)}, 'items': [{k: r[k] for k in ('question', 'answer', 'start', 'end', 'score', 'n_windows', 'seconds')} for r in results], 'golds': golds, 'frozen_report': frozen_form, 'adapted_report': adapted_form},
    'comparison': comparison,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/layoutlm_document_qa_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen model reads most receipt totals already — its test ANLS sits well above both non-neural baselines — and a bounded fine-tuning of the last four encoder blocks and the span head on 119 receipts lifts held-out ANLS by several points, concentrated in the fields it had been missing (item names, quantities, cash and change lines), in about a quarter of an hour on CPU, with a 113 MB adapter that reloads to identical answers. That is the claim: the adaptation contract works end to end on a real gold-span document corpus whose OCR the model never sees as pixels, and the numbers it produces are read against two non-neural baselines and the frozen model rather than in isolation.

The test split is 229 questions over 50 receipts from one seeded split of one corpus, the questions are templated from field categories rather than written by people, the metrics are two reference-based scores (own implementations of DocVQA's normalisation, neither a human judgement), and the OCR is CORD's annotation — clean words in reading order with tight boxes, which no OCR engine on a photographed receipt reproduces. So a gain here says the contract works, not that the adapted model is better on your documents or your OCR, that it handles other languages, handwriting or long pages, or that its spans are faithful — a reader returns a plausible wrong span for an unanswerable question, and the adaptation changes nothing about that. Fine-tuning on a narrow corpus can also erode the model elsewhere; the invoice re-read in Section 9 is one page of evidence about that, not a measurement.

Three things to carry to real data. **Baselines first:** the last-number and keyword-lookup baselines and the frozen model's score on *your* gold spans, with *your* OCR, are the numbers to read before any adapted one, per field. **Leakage:** keep every question on a page in one split (the contract does this) and split by document or by vendor when your pages come from few sources, never at random over near-duplicate pages — CORD-v2 itself contains one duplicated receipt, which the sample drops. **Ceilings:** a question + page over one 512-token window is answered window by window at inference and dropped from training by the fit check; long-document training is out of scope.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify one column of a real annotated corpus, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded fine-tuning, evaluate against two trivial baselines and the frozen model on a page-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, document-understanding accuracy on any other domain or OCR, a usable rejection threshold, or production fitness.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_ENCODER_LAYERS = 2` and compare the artifact size and the test scores; raise `EPOCHS` and watch the validation ANLS pick the epoch; ask the adapted model `What is the delivery address?` on the invoice and read the score of a span the page cannot support; or bring your own annotated pages through BYOD and read the two baselines before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/layoutlm-document-qa-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/layoutlm-document-qa-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/layoutlm-document-qa-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model (Impira, MIT): https://huggingface.co/impira/layoutlm-document-qa
- Upstream architecture code (LayoutLM, Microsoft): https://github.com/microsoft/unilm/tree/master/layoutlm
- LayoutLM: Pre-training of Text and Layout for Document Image Understanding (Xu et al., 2019): https://arxiv.org/abs/1912.13318
- DocVQA: A Dataset for VQA on Document Images (Mathew, Karatzas, Jawahar, 2020): https://arxiv.org/abs/2007.00398
- Scene Text Visual Question Answering — the ANLS metric (Biten et al., 2019): https://arxiv.org/abs/1905.13648
- CORD: A Consolidated Receipt Dataset for Post-OCR Parsing (Park et al., NeurIPS 2019 Document Intelligence Workshop; CORD-v2, CC BY 4.0): https://huggingface.co/datasets/naver-clova-ix/cord-v2
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)